# Exercice 23b : SimCLR avec ResNet-18 sur CIFAR-10 complet (Colab, GPU T4)

**Chapitre lié** : 23. Apprentissage contrastif.

**Objectif.** Refaire l'exercice 23a à l'échelle où SimCLR commence à donner sa mesure :
un ResNet-18 adapté aux images 32×32, les 50 000 images d'entraînement **sans étiquette**,
100 époques, puis une sonde linéaire sur l'encodeur gelé. Comparer avec la version locale
(petit CNN, 25 000 images, 15 époques) et avec un ResNet-18 au hasard.

**Perte optimisée** : InfoNCE dans sa version SimCLR (NT-Xent), température τ = 0,5.
Variante : `PERTE = "barlow"` (Barlow Twins, sans négatifs), tout le reste identique.

| Réglage | Valeur |
|---|---|
| Encodeur | ResNet-18 (torchvision), 1re convolution 3×3 de pas 1, sans max pooling initial (comme SimCLR, annexe B.9) |
| Tête de projection | 512 → 2 048 → 128 (BatchNorm, ReLU) |
| Lot | 512 images, donc 1 024 vues et 1 023 candidates par vue |
| Optimiseur | AdamW, taux 10⁻³ avec montée puis décroissance (« one cycle »), décroissance des poids 10⁻⁴ |
| Augmentation | celle de SimCLR (recadrage 20-100 %, retournement, couleur p = 0,8, gris p = 0,2), **calculée sur le GPU image par image** |
| Précision | float16 (`torch.autocast` + `GradScaler`) : le T4 ne fait pas de bfloat16 |

**Pourquoi l'augmentation sur le GPU ?** Colab gratuit ne donne que 2 cœurs de CPU. Les
augmentations de torchvision, appliquées image par image sur le CPU, deviendraient le goulot
(100 000 vues par époque à 2 vues par image et 50 000 images). Le notebook les recalcule donc
en PyTorch pur sur le GPU, **avec des paramètres tirés indépendamment pour chaque image**
(une transformation torchvision appliquée à un lot tirerait le même recadrage pour toutes).

In [ ]:
# 1. Vérification du GPU : ce notebook est écrit pour un GPU NVIDIA T4 (Colab gratuit, ou Kaggle « GPU T4 x2 »)
!nvidia-smi
import os
import torch
print("PyTorch", torch.__version__, "| CUDA disponible :", torch.cuda.is_available())
if not torch.cuda.is_available() and os.environ.get("IA_TEST_LOCAL") != "1":
    raise SystemExit("Pas de GPU : Exécution › Modifier le type d'exécution › GPU T4, puis tout relancer. "
                     "Sur CPU, une époque prendrait plus d'une heure.")

In [ ]:
# 2. Dépendances : rien à installer. torch, torchvision, numpy, matplotlib, pyarrow,
# huggingface_hub et Pillow sont déjà dans l'image Colab (on n'épingle pas torch : cela réinstallerait ~1 Go).
import json
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
import torchvision
from torch import nn

# IA_TEST_LOCAL=1 : petite version (2 048 images, 2 époques) pour vérifier le code hors Colab
TEST_LOCAL = os.environ.get("IA_TEST_LOCAL") == "1"
SEED = 42
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
FP16 = device == "cuda"   # demi-précision float16 sur le T4 (jamais bfloat16)

CONFIG = dict(
    PERTE="infonce",          # "infonce" (SimCLR) ou "barlow" (Barlow Twins)
    EPOQUES=2 if TEST_LOCAL else 100,   # 50 pour une session plus courte
    LOT=256 if TEST_LOCAL else 512,
    TAU=0.5,                  # température de NT-Xent
    LAMBDA_BARLOW=5e-3,       # poids hors diagonale de Barlow Twins
    LR_MAX=1e-3,
    DECROISSANCE=1e-4,
    N_IMAGES=2048 if TEST_LOCAL else 50000,
    N_TEST=1000 if TEST_LOCAL else 10000,
    KNN_TOUTES_LES=1 if TEST_LOCAL else 10,   # suivi kNN toutes les 10 époques
    N_RARES=500,              # scénario « peu d'étiquettes » : 50 par classe
)


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
print("Périphérique :", device, "| float16 :", FP16)
print(CONFIG)

In [ ]:
# 3. Données : CIFAR-10 (Krizhevsky, 2009), copie Hugging Face `uoft-cs/cifar10` (deux fichiers
# parquet, ~120 Mo, rapide depuis Colab), gardé en uint8 SUR LE GPU (50 000 × 3 × 32 × 32 octets
# = 150 Mo) : plus aucun transfert CPU → GPU pendant l'entraînement.
# Secours : le serveur d'origine via torchvision (lent selon les jours).
import io

CLASSES = ["avion", "automobile", "oiseau", "chat", "cerf", "chien", "grenouille", "cheval", "bateau", "camion"]


def charger_cifar10(split):
    try:
        import pyarrow.parquet as pq
        from huggingface_hub import hf_hub_download
        from PIL import Image
        f = hf_hub_download("uoft-cs/cifar10", f"plain_text/{split}-00000-of-00001.parquet", repo_type="dataset")
        t = pq.read_table(f)
        x = np.stack([np.asarray(Image.open(io.BytesIO(d["bytes"])).convert("RGB")) for d in t.column("img").to_pylist()])
        y = np.asarray(t.column("label").to_pylist())
    except Exception as e:  # pas d'accès au Hub : serveur d'origine
        print("Hugging Face indisponible (", e, ") : téléchargement depuis Toronto")
        ds = torchvision.datasets.CIFAR10("data", train=(split == "train"), download=True)
        x, y = ds.data, np.asarray(ds.targets)
    return x, y


def vers_tenseur(x, y, n):
    x = torch.from_numpy(x[:n]).permute(0, 3, 1, 2).contiguous()   # N × 3 × 32 × 32, uint8
    return x.to(device), torch.from_numpy(y[:n]).long().to(device)


X_train, y_train = vers_tenseur(*charger_cifar10("train"), CONFIG["N_IMAGES"])
X_test, y_test = vers_tenseur(*charger_cifar10("test"), CONFIG["N_TEST"])
MOY = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1, 3, 1, 1)
ET = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1, 3, 1, 1)


def normaliser(x_uint8):
    return (x_uint8.float() / 255 - MOY) / ET


print("Entraînement :", tuple(X_train.shape), "| test :", tuple(X_test.shape))

In [ ]:
# 4. Augmentation de SimCLR sur le GPU, paramètres tirés IMAGE PAR IMAGE.
# Mêmes réglages que ia_commun.augmentations.AugmentationSimCLR (exercice 23a) :
# recadrage redimensionné (20 à 100 % de la surface, rapport 3/4 à 4/3), retournement (p = 0,5),
# distorsion de couleur de force 0,5 avec p = 0,8 (luminosité, contraste, saturation ± 0,4, teinte ± 0,1),
# niveaux de gris (p = 0,2).
# En simplifiant par rapport à torchvision : (1) un recadrage trop large est rogné au lieu d'être
# retiré au sort ; (2) les quatre réglages de couleur sont appliqués dans un ordre fixe ;
# (3) la teinte tourne dans l'espace YIQ au lieu de HSV. Les vues obtenues sont du même type.
POIDS_GRIS = torch.tensor([0.299, 0.587, 0.114], device=device).view(1, 3, 1, 1)


def gris(x):
    return (x * POIDS_GRIS).sum(1, keepdim=True)


def augmenter(x_uint8):
    n = x_uint8.shape[0]
    x = x_uint8.float() / 255
    u = lambda *forme: torch.rand(*forme, device=device)   # noqa: E731
    # Recadrage redimensionné : on choisit la fenêtre (en coordonnées [-1, 1]) puis grid_sample
    surface = 0.2 + 0.8 * u(n)
    rapport = torch.exp(math.log(3 / 4) + (math.log(4 / 3) - math.log(3 / 4)) * u(n))
    largeur = torch.sqrt(surface * rapport).clamp(max=1.0)
    hauteur = torch.sqrt(surface / rapport).clamp(max=1.0)
    cx = (2 * u(n) - 1) * (1 - largeur)
    cy = (2 * u(n) - 1) * (1 - hauteur)
    signe = torch.where(u(n) < 0.5, -1.0, 1.0)                # retournement horizontal
    theta = torch.zeros(n, 2, 3, device=device)
    theta[:, 0, 0], theta[:, 0, 2] = largeur * signe, cx
    theta[:, 1, 1], theta[:, 1, 2] = hauteur, cy
    grille = F.affine_grid(theta, (n, 3, 32, 32), align_corners=False)
    x = F.grid_sample(x, grille, mode="bilinear", padding_mode="reflection", align_corners=False)
    # Distorsion de couleur, appliquée à 80 % des images
    actif = (u(n) < 0.8).float().view(n, 1, 1, 1)
    def facteur(amplitude):
        return (1 + amplitude * (2 * u(n) - 1)).view(n, 1, 1, 1)
    y = (x * facteur(0.4)).clamp(0, 1)                                          # luminosité
    m = gris(y).mean(dim=(2, 3), keepdim=True)
    y = ((y - m) * facteur(0.4) + m).clamp(0, 1)                                # contraste
    g = gris(y)
    y = ((y - g) * facteur(0.4) + g).clamp(0, 1)                                # saturation
    angle = (2 * math.pi * 0.1 * (2 * u(n) - 1)).view(n, 1, 1)                  # teinte (± 0,1 tour)
    yiq = torch.tensor([[0.299, 0.587, 0.114], [0.596, -0.274, -0.322], [0.211, -0.523, 0.312]], device=device)
    v = torch.einsum("ij,njhw->nihw", yiq, y)
    i2 = v[:, 1] * torch.cos(angle) - v[:, 2] * torch.sin(angle)
    q2 = v[:, 1] * torch.sin(angle) + v[:, 2] * torch.cos(angle)
    v = torch.stack([v[:, 0], i2, q2], 1)
    y = torch.einsum("ij,njhw->nihw", torch.linalg.inv(yiq), v).clamp(0, 1)
    x = actif * y + (1 - actif) * x
    # Niveaux de gris, 20 % des images
    en_gris = (u(n) < 0.2).float().view(n, 1, 1, 1)
    x = en_gris * gris(x).expand(-1, 3, -1, -1) + (1 - en_gris) * x
    return (x - MOY) / ET


# Contrôle visuel : 8 images, deux vues chacune
torch.manual_seed(SEED)
apercu = X_train[:8]
v1, v2 = augmenter(apercu), augmenter(apercu)
fig, axes = plt.subplots(3, 8, figsize=(12, 4.8))
for k in range(8):
    for ligne, img in enumerate((normaliser(apercu)[k], v1[k], v2[k])):
        axes[ligne, k].imshow((img * ET[0] + MOY[0]).clamp(0, 1).permute(1, 2, 0).cpu())
        axes[ligne, k].axis("off")
axes[0, 0].set_title("original", fontsize=8, loc="left")
axes[1, 0].set_title("vue 1", fontsize=8, loc="left")
axes[2, 0].set_title("vue 2", fontsize=8, loc="left")
plt.tight_layout()
plt.show()

In [ ]:
# 5. Modèle : ResNet-18 adapté à CIFAR-10 (encodeur f, h de 512 nombres) + tête de projection g
def creer_encodeur():
    r = torchvision.models.resnet18(weights=None)
    r.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)   # 7×7 pas 2 → 3×3 pas 1
    r.maxpool = nn.Identity()                                       # pas de max pooling initial
    r.fc = nn.Identity()                                            # sortie : h, 512 nombres
    return r


class ModeleSSL(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = creer_encodeur()
        self.g = nn.Sequential(nn.Linear(512, 2048), nn.BatchNorm1d(2048), nn.ReLU(), nn.Linear(2048, 128))

    def forward(self, x):
        h = self.f(x)
        return h, self.g(h)


def nt_xent(z1, z2, tau):
    # Entropie croisée sur les 2N vues : la « classe » de la vue i est sa jumelle (i ± N)
    z = F.normalize(torch.cat([z1, z2]).float(), dim=1)   # calcul en float32 même sous autocast
    n = z1.shape[0]
    sim = (z @ z.T / tau).masked_fill(torch.eye(2 * n, dtype=torch.bool, device=z.device), float("-inf"))
    cibles = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, cibles)


def perte_barlow(z1, z2, lam):
    z1, z2 = z1.float(), z2.float()
    n = z1.shape[0]
    z1 = (z1 - z1.mean(0)) / (z1.std(0, unbiased=False) + 1e-5)
    z2 = (z2 - z2.mean(0)) / (z2.std(0, unbiased=False) + 1e-5)
    c = z1.T @ z2 / n
    d = torch.diagonal(c)
    return ((1 - d) ** 2).sum() + lam * ((c ** 2).sum() - (d ** 2).sum())


def perte(z1, z2):
    if CONFIG["PERTE"] == "barlow":
        return perte_barlow(z1, z2, CONFIG["LAMBDA_BARLOW"])
    return nt_xent(z1, z2, CONFIG["TAU"])


@torch.no_grad()
def representations(modele, X, lot=1000):
    modele.eval()
    sorties = []
    for i in range(0, len(X), lot):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=FP16):
            h, _ = modele(normaliser(X[i:i + lot]))
        sorties.append(h.float())
    return torch.cat(sorties)


@torch.no_grad()
def knn(h_train, y_tr, h_eval, y_ev, k=20):
    a, b = F.normalize(h_train, dim=1), F.normalize(h_eval, dim=1)
    pred = torch.cat([torch.mode(y_tr[(b[i:i + 1000] @ a.T).topk(k, dim=1).indices], dim=1).values
                      for i in range(0, len(b), 1000)])
    return (pred == y_ev).float().mean().item()


def sonde_lineaire(h_train, y_tr, h_eval, y_ev, pas=500, decroissance=1e-4):
    moy, et = h_train.mean(0), h_train.std(0) + 1e-6
    a, b = (h_train - moy) / et, (h_eval - moy) / et
    torch.manual_seed(SEED)
    lin = nn.Linear(a.shape[1], 10).to(device)
    opt = torch.optim.Adam(lin.parameters(), lr=1e-2, weight_decay=decroissance)
    for _ in range(pas):
        p = F.cross_entropy(lin(a), y_tr)
        opt.zero_grad()
        p.backward()
        opt.step()
    with torch.no_grad():
        return (lin(b).argmax(1) == y_ev).float().mean().item()


@torch.no_grad()
def ecart_type_z(modele, X):
    # 0 = effondrement (toutes les sorties identiques), ≈ 1 = bien réparties (repère de SimSiam)
    modele.eval()
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=FP16):
        _, z = modele(normaliser(X))
    z = F.normalize(z.float(), dim=1)
    return (z.std(0).mean() * z.shape[1] ** 0.5).item()


seed_everything()
modele = ModeleSSL().to(device)
print(f"Encodeur : {sum(p.numel() for p in modele.f.parameters()):,} paramètres".replace(",", " "))

In [ ]:
# 6. Référence : le même ResNet-18 aux poids aléatoires (jamais entraîné)
seed_everything()
hasard = ModeleSSL().to(device)
h_tr, h_te = representations(hasard, X_train), representations(hasard, X_test)
resultats = {"config": CONFIG, "hasard": {
    "sonde": sonde_lineaire(h_tr, y_train, h_te, y_test),
    "knn": knn(h_tr, y_train, h_te, y_test),
    "ecart_type": ecart_type_z(hasard, X_test[:1024]),
}}
del hasard, h_tr, h_te
print("ResNet-18 au hasard :", resultats["hasard"])

In [ ]:
# 7. Entraînement auto-supervisé (les étiquettes y_train ne sont PAS utilisées ici)
# Point de reprise : si la session est coupée, relancer cette cellule reprend à la dernière
# époque sauvegardée (fichier simclr_23b.pt, à copier sur Drive pour survivre à une déconnexion).
seed_everything()
modele = ModeleSSL().to(device)
lots_par_epoque = CONFIG["N_IMAGES"] // CONFIG["LOT"]
optim = torch.optim.AdamW(modele.parameters(), lr=CONFIG["LR_MAX"], weight_decay=CONFIG["DECROISSANCE"])
sched = torch.optim.lr_scheduler.OneCycleLR(optim, max_lr=CONFIG["LR_MAX"],
                                            total_steps=CONFIG["EPOQUES"] * lots_par_epoque)
scaler = torch.amp.GradScaler("cuda", enabled=FP16)
hist = {"perte": [], "ecart_type": [], "knn": {}, "duree": []}
debut = 0
if os.path.exists("simclr_23b.pt"):
    etat = torch.load("simclr_23b.pt", map_location=device)
    if etat["config"] == CONFIG:
        modele.load_state_dict(etat["modele"]); optim.load_state_dict(etat["optim"])
        sched.load_state_dict(etat["sched"]); scaler.load_state_dict(etat["scaler"])
        hist, debut = etat["hist"], etat["epoque"]
        print("Reprise à l'époque", debut)

g = torch.Generator().manual_seed(SEED)   # ordre des lots reproductible
t0 = time.perf_counter() - (hist["duree"][-1] if hist["duree"] else 0)
for ep in range(debut, CONFIG["EPOQUES"]):
    modele.train()
    ordre = torch.randperm(CONFIG["N_IMAGES"], generator=g).to(device)
    somme = 0.0
    for b in range(lots_par_epoque):
        xb = X_train[ordre[b * CONFIG["LOT"]:(b + 1) * CONFIG["LOT"]]]
        vues = torch.cat([augmenter(xb), augmenter(xb)])     # deux tirages indépendants par image
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=FP16):
            _, z = modele(vues)
        z1, z2 = z.chunk(2)
        p = perte(z1, z2)
        optim.zero_grad(set_to_none=True)
        scaler.scale(p).backward()
        scaler.step(optim)
        scaler.update()
        sched.step()
        somme += p.item()
    hist["perte"].append(somme / lots_par_epoque)
    hist["ecart_type"].append(ecart_type_z(modele, X_test[:1024]))
    hist["duree"].append(time.perf_counter() - t0)
    ligne = (f"époque {ep + 1:3d}/{CONFIG['EPOQUES']} : perte {hist['perte'][-1]:.4f}, "
             f"écart-type × √d {hist['ecart_type'][-1]:.3f}, {hist['duree'][-1] / 60:.1f} min")
    if (ep + 1) % CONFIG["KNN_TOUTES_LES"] == 0 or ep + 1 == CONFIG["EPOQUES"]:
        hist["knn"][ep + 1] = knn(representations(modele, X_train), y_train, representations(modele, X_test), y_test)
        ligne += f", kNN {hist['knn'][ep + 1]:.1%}"
    if ep == debut:
        ligne += f" (durée totale prévue ≈ {hist['duree'][-1] * (CONFIG['EPOQUES'] - debut) / 60:.0f} min)"
    print(ligne)
    torch.save({"modele": modele.state_dict(), "optim": optim.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "hist": hist, "epoque": ep + 1, "config": CONFIG},
               "simclr_23b.pt")
resultats["entrainement"] = hist

In [ ]:
# 8. Évaluation : sonde linéaire (50 000 étiquettes), kNN, et scénario « 500 étiquettes »
h_tr, h_te = representations(modele, X_train), representations(modele, X_test)
gen = np.random.default_rng(SEED)
y_np = y_train.cpu().numpy()
rares = torch.from_numpy(np.concatenate([
    gen.choice(np.flatnonzero(y_np == c), size=CONFIG["N_RARES"] // 10, replace=False) for c in range(10)
])).to(device)
resultats["ssl"] = {
    "sonde": sonde_lineaire(h_tr, y_train, h_te, y_test),
    "knn": knn(h_tr, y_train, h_te, y_test),
    "sonde_500": sonde_lineaire(h_tr[rares], y_train[rares], h_te, y_test, decroissance=1e-3),
    "ecart_type": ecart_type_z(modele, X_test[:1024]),
}
print(f"{CONFIG['PERTE']} : sonde linéaire {resultats['ssl']['sonde']:.1%} "
      f"(au hasard : {resultats['hasard']['sonde']:.1%}), kNN {resultats['ssl']['knn']:.1%}, "
      f"sonde avec 500 étiquettes {resultats['ssl']['sonde_500']:.1%}")

In [ ]:
# 9. Bilan, courbes et sauvegarde
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
ep = np.arange(1, len(hist["perte"]) + 1)
a1.plot(ep, hist["perte"])
a1.set(xlabel="époque", ylabel="perte", title=f"Perte ({CONFIG['PERTE']})")
a2.plot([int(k) for k in hist["knn"]], [100 * v for v in hist["knn"].values()], "o-")
a2.axhline(100 * resultats["hasard"]["knn"], color="gray", ls=":", label="ResNet-18 au hasard")
a2.set(xlabel="époque", ylabel="exactitude kNN en test (%)", title="Suivi kNN (k = 20)")
a2.legend()
plt.tight_layout()
plt.savefig("courbes_23b.png", dpi=120)
plt.show()

with open("resultats_23b.json", "w") as f:
    json.dump(resultats, f, ensure_ascii=False, indent=1, default=float)
print(json.dumps({k: v for k, v in resultats.items() if k != "entrainement"}, ensure_ascii=False, indent=1))

if not TEST_LOCAL:
    # Seuils prudents, NON mesurés sur T4 : à confirmer au premier lancement (voir README)
    assert resultats["ssl"]["sonde"] >= 0.75, "sonde linéaire sous 75 %"
    assert resultats["ssl"]["sonde"] - resultats["hasard"]["sonde"] >= 0.25, "gain sur le hasard sous 25 points"
    assert resultats["ssl"]["ecart_type"] >= 0.5, "écart-type faible : effondrement ?"

In [ ]:
# 10. Télécharger les résultats avant la fin de la session (Colab) ; sur Kaggle : onglet Output
try:
    from google.colab import files
    for nom in ("resultats_23b.json", "courbes_23b.png", "simclr_23b.pt"):
        files.download(nom)
except ImportError:
    print("Hors Colab : fichiers dans", os.getcwd())